In [1]:
# Ensure input CSV is valid
import pandas as pd

# Input Dataframe
input_path = "~/MedPAIR/Dataset/Datasets/equipair/household_income.csv"
df = pd.read_csv(input_path)

# Output file path
output_path = "~/MedPAIR/Results/qwen/equipair/household_income.csv"

print("Finished reading input and output files")

Finished reading input and output files


In [2]:
# Initialize Model

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "Qwen/Qwen2.5-72B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Finished creating model")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/37 [00:00<?, ?it/s]

Finished creating model


In [3]:
# Small Query Test
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen. You are a helpful medical assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


A large language model (LLM) is an advanced type of artificial intelligence designed to understand, generate, and interact with human language in a sophisticated manner. These models are typically based on deep learning architectures, such as transformers, and are trained on vast amounts of text data from the internet, books, and other sources. The goal of LLMs is to capture the nuances of human language, including syntax, semantics, and context, allowing them to perform a wide range of tasks, such as:

1. **Text Generation**: Creating coherent and contextually relevant text, such as articles, stories, or responses to user queries.
2. **Translation**: Translating text from one language to another with high accuracy.
3. **Summarization**: Condensing long documents into shorter, more manageable summaries.
4. **Question Answering**: Providing answers to specific questions based on the information available in the text.
5. **Conversational AI**: Engaging in natural and meaningful conversat

In [4]:
import os
import re
import time

processed_ids = set()

# Create output file with header if it doesn't exist
if not os.path.exists(output_path):
    header_cols = df.columns.tolist() + ['Raw_Response', 'LLM_answer'] + [f"label_{i+1}" for i in range(30)]
    pd.DataFrame(columns=header_cols).to_csv(output_path, index=False)
else:
    try:
        existing_df = pd.read_csv(output_path)
        processed_ids_arr = existing_df["ID_corr"].unique()
        for id in processed_ids_arr:
            processed_ids.add(str(id))
    except:
        pass

# Iterate through each row
for i in range(len(df)): 
    row_id = str(df.loc[i, "ID_corr"])
    if row_id in processed_ids:
        print(f"Found {row_id} in existing processed IDs. Skipping.")
        continue

    try:
        formatted_sentences = df.loc[i, "original_sentences"]
        options = df.loc[i, "question_options"]

        prompt = f"""
You are given a list of sentences from a clinical vignette and a multiple-choice clinical question. 

Your task is twofold:
(1) Select the most appropriate answer from the given options.
(2) Label each sentence as either [High Relevance], [Low Relevance], or [Irrelevant], based on its contribution to answering the question.

Definitions:
[High Relevance]: Sentences that directly support the correct answer with essential clinical information (e.g., diagnosis, key test results).
[Low Relevance]: Sentences that provide useful context or background but are not critical to answering.
[Irrelevant]: Sentences unrelated to the question or not useful for reasoning.

Question and Options:
{options.strip()}

Sentences:
{formatted_sentences}

Please provide your answer selection first (e.g., "Answer: B"), followed by the relevance label for each sentence in order.
""".strip()

        # Use chat template format (like your working example)
        messages = [
            {"role": "system", "content": "You are Qwen. You are a helpful medical assistant."},
            {"role": "user", "content": prompt}
        ]
        
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        # Generate prediction
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Extract only the generated part (not the input)
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        # Decode the response
        raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # Extract Qwen answer
        answer_match = re.search(r"Answer:\s*([A-J])", raw_response, re.IGNORECASE)
        answer = answer_match.group(1).upper() if answer_match else None

        # Extract sentence-level relevance labels
        labels = re.findall(r"\[\s*(High Relevance|Low Relevance|Irrelevant)\s*\]", raw_response, re.IGNORECASE)
        labels = [label.title() for label in labels]
        labels = labels[:30] + [None] * (30 - len(labels))  

    except Exception as e:
        print(f"Error at row {i}: {e}")
        raw_response = ""
        answer = None
        labels = [None] * 30

    row_data = pd.concat(
        [df.iloc[[i]].reset_index(drop=True),
         pd.DataFrame([[raw_response, answer] + labels], columns=['Raw_Response', 'LLM_answer'] + [f"label_{j+1}" for j in range(30)])
        ],
        axis=1
    )
    row_data.to_csv(output_path, mode='a', index=False, header=False)

    print(f"Row {i} processed and saved.")
    time.sleep(0.2)

Row 0 processed and saved.
Row 1 processed and saved.
Row 2 processed and saved.
Row 3 processed and saved.
Row 4 processed and saved.
Row 5 processed and saved.
Row 6 processed and saved.
Row 7 processed and saved.
Row 8 processed and saved.
Row 9 processed and saved.
Row 10 processed and saved.
Row 11 processed and saved.
Row 12 processed and saved.
Row 13 processed and saved.
Row 14 processed and saved.
Row 15 processed and saved.
Row 16 processed and saved.
Row 17 processed and saved.
Row 18 processed and saved.
Row 19 processed and saved.
Row 20 processed and saved.
Row 21 processed and saved.
Row 22 processed and saved.
Row 23 processed and saved.
Row 24 processed and saved.
Row 25 processed and saved.
Row 26 processed and saved.
Row 27 processed and saved.
Row 28 processed and saved.
Row 29 processed and saved.
Row 30 processed and saved.
Row 31 processed and saved.
Row 32 processed and saved.
Row 33 processed and saved.
Row 34 processed and saved.
Row 35 processed and saved.
Ro

Row 287 processed and saved.
Row 288 processed and saved.
Row 289 processed and saved.
Row 290 processed and saved.
Row 291 processed and saved.
Row 292 processed and saved.
Row 293 processed and saved.
Row 294 processed and saved.
Row 295 processed and saved.
Row 296 processed and saved.
Row 297 processed and saved.
Row 298 processed and saved.
Row 299 processed and saved.
Row 300 processed and saved.
Row 301 processed and saved.
Row 302 processed and saved.
Row 303 processed and saved.
Row 304 processed and saved.
Row 305 processed and saved.
Row 306 processed and saved.
Row 307 processed and saved.
Row 308 processed and saved.
Row 309 processed and saved.
Row 310 processed and saved.
Row 311 processed and saved.
Row 312 processed and saved.
Row 313 processed and saved.
Row 314 processed and saved.
Row 315 processed and saved.
Row 316 processed and saved.
Row 317 processed and saved.
Row 318 processed and saved.
Row 319 processed and saved.
Row 320 processed and saved.
Row 321 proces

Row 570 processed and saved.
Row 571 processed and saved.
Row 572 processed and saved.
Row 573 processed and saved.
Row 574 processed and saved.
Row 575 processed and saved.
Row 576 processed and saved.
Row 577 processed and saved.
Row 578 processed and saved.
Row 579 processed and saved.
Row 580 processed and saved.
Row 581 processed and saved.
Row 582 processed and saved.
Row 583 processed and saved.
Row 584 processed and saved.
Row 585 processed and saved.
Row 586 processed and saved.
Row 587 processed and saved.
Row 588 processed and saved.
Row 589 processed and saved.
Row 590 processed and saved.
Row 591 processed and saved.
Row 592 processed and saved.
Row 593 processed and saved.
Row 594 processed and saved.
Row 595 processed and saved.
Row 596 processed and saved.
Row 597 processed and saved.
Row 598 processed and saved.
Row 599 processed and saved.
Row 600 processed and saved.
Row 601 processed and saved.
Row 602 processed and saved.
Row 603 processed and saved.
Row 604 proces

Row 853 processed and saved.
Row 854 processed and saved.
Row 855 processed and saved.
Row 856 processed and saved.
Row 857 processed and saved.
Row 858 processed and saved.
Row 859 processed and saved.
Row 860 processed and saved.
Row 861 processed and saved.
Row 862 processed and saved.
Row 863 processed and saved.
Row 864 processed and saved.
Row 865 processed and saved.
Row 866 processed and saved.
Row 867 processed and saved.
Row 868 processed and saved.
Row 869 processed and saved.
Row 870 processed and saved.
Row 871 processed and saved.
Row 872 processed and saved.
Row 873 processed and saved.
Row 874 processed and saved.
Row 875 processed and saved.
Row 876 processed and saved.
Row 877 processed and saved.
Row 878 processed and saved.
Row 879 processed and saved.
Row 880 processed and saved.
Row 881 processed and saved.
Row 882 processed and saved.
Row 883 processed and saved.
Row 884 processed and saved.
Row 885 processed and saved.
Row 886 processed and saved.
Row 887 proces

Row 1131 processed and saved.
Row 1132 processed and saved.
Row 1133 processed and saved.
Row 1134 processed and saved.
Row 1135 processed and saved.
Row 1136 processed and saved.
Row 1137 processed and saved.
Row 1138 processed and saved.
Row 1139 processed and saved.
Row 1140 processed and saved.
Row 1141 processed and saved.
Row 1142 processed and saved.
Row 1143 processed and saved.
Row 1144 processed and saved.
Row 1145 processed and saved.
Row 1146 processed and saved.
Row 1147 processed and saved.
Row 1148 processed and saved.
Row 1149 processed and saved.
Row 1150 processed and saved.
Row 1151 processed and saved.
Row 1152 processed and saved.
Row 1153 processed and saved.
Row 1154 processed and saved.
Row 1155 processed and saved.
Row 1156 processed and saved.
Row 1157 processed and saved.
Row 1158 processed and saved.
Row 1159 processed and saved.
Row 1160 processed and saved.
Row 1161 processed and saved.
Row 1162 processed and saved.
Row 1163 processed and saved.
Row 1164 p